# 📡 Conocimiento estático y búsqueda web

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 1 — 35 minutos**

---

## Objetivo

Entender por qué los LLMs "no saben" cosas recientes — y ver la primera estrategia para solucionarlo.

## Al terminar este bloque vas a poder:

1. Explicar el concepto de *cutoff date* y sus consecuencias prácticas.
2. Conectar una API de búsqueda web para inyectar información actual en un prompt.
3. Construir un pipeline sencillo: pregunta → búsqueda → respuesta fundamentada.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Conocimiento estático** | Todo lo que el LLM aprendió durante entrenamiento — congelado en el tiempo. |
| **Cutoff date** | La fecha hasta la cual el modelo vio datos. Después de ahí, "no sabe" nada nuevo. |
| **Alucinación** | Cuando el modelo genera información plausible pero falsa o desactualizada con total confianza. |
| **Web Search Integration** | Patrón que consulta la web antes del LLM para darle contexto fresco. |
| **RAG** | Retrieval-Augmented Generation — el nombre formal del patrón que une recuperación + generación. |

## El problema: los LLMs tienen memoria congelada

### Analogía

Un LLM es como un experto que estudió muchísimo hasta cierta fecha, se encerró en un cuarto sin internet, y desde ahí responde preguntas. Todo lo que pasó después de que cerró la puerta, simplemente no lo sabe — y si le preguntás, va a inventar algo que suene razonable.

### Dónde vive esto en el mundo real

- Gemini y GPT-4 tienen cutoff dates. Si les preguntás sobre eventos muy recientes, pueden alucinar.
- Perplexity, Bing Chat y ChatGPT con búsqueda web resuelven esto conectando un motor de búsqueda antes de responder.
- En sistemas RAG empresariales (lo que vamos a construir hoy), el "buscador" no es Google sino tu propia base de documentos.

### El patrón de solución

```
[Pregunta del usuario]
        ↓
[Búsqueda web / base de documentos]
        ↓
[Contexto actualizado → LLM]
        ↓
[Respuesta fundamentada]
```

Hoy vamos a ver la versión con búsqueda web. En los próximos bloques, el buscador va a ser ChromaDB con tus propios documentos.

### ✎ Para pensar

- ¿Qué diferencia hay entre que el modelo "no sepa" algo y que lo "invente"? ¿Cuál es más peligroso?
- Si el modelo tiene acceso a búsqueda web, ¿podría buscar información errónea y tomarla como verdadera?

## Configuración

Necesitás dos API keys:
- **SERP API**: busca resultados de Google de forma programática
- **Gemini API**: genera la respuesta final usando Gemini

Las keys se guardan en el archivo `.env`. **Nunca las pongas directamente en el código ni las subas a git.**

In [ ]:
# Instalamos las librerías
# google-search-results: cliente para SerpAPI (búsqueda web)
# google-genai: SDK oficial de Google para Gemini
!uv pip install google-search-results google-genai python-dotenv -q
print("✓ Librerías instaladas correctamente")

In [ ]:
import os
import json
from dataclasses import dataclass
from dotenv import load_dotenv
from google import genai

load_dotenv()

SERP_API_KEY = os.getenv("SERPAPI_API_KEY", "").strip()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()

if not SERP_API_KEY:
    raise ValueError("Falta configurar SERPAPI_API_KEY en el archivo .env")
if not GEMINI_API_KEY:
    raise ValueError("Falta configurar GEMINI_API_KEY en el archivo .env")

# Creamos el cliente de Gemini una sola vez — lo reutilizamos en todo el notebook
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

print("✓ APIs configuradas correctamente")
print("  SERP API: lista para búsqueda web")
print("  Gemini: cliente inicializado")

## Paso 1 — Buscar en la web

La función `serp_results()` hace una búsqueda en Google a través de SERP API y devuelve los snippets de los primeros resultados. Cada resultado tiene URL, título y un fragmento de texto.

In [ ]:
from serpapi import GoogleSearch

@dataclass(frozen=True)
class SnippetCitation:
    Url: str
    Title: str
    Snippet: str

def serp_results(query: str, num=5, api_key=SERP_API_KEY):
    params = {
        "engine": "google",
        "q": query,
        "api_key": api_key,
        "num": num
    }
    search = GoogleSearch(params)
    resultado_busqueda = search.get_dict()
    organic_results = resultado_busqueda.get("organic_results", [])
    resultados = []
    for result in organic_results:
        try:
            resultados.append(SnippetCitation(
                Url=result["link"],
                Title=result["title"],
                Snippet=result["snippet"]
            ))
        except Exception as e:
            print(e)
    return resultados

In [ ]:
serp_res = serp_results("¿Cual es el nombre del Papa?")
serp_res

## Paso 2 — Construir el prompt con contexto

El truco es simple: tomamos los snippets de búsqueda y los pegamos en el prompt como "contexto". El modelo tiene instrucciones de responder **solo** con lo que está en ese contexto — y de decir que no sabe si la respuesta no está.

In [ ]:
def responder_con_gemini(prompt, temperature=0, model="gemini-2.5-flash"):
    """
    Llama a Gemini con un prompt y devuelve la respuesta como texto.

    Parámetros:
    -----------
    prompt : str
        El prompt completo (sistema + contexto + pregunta ya armados)
    temperature : float
        Creatividad del modelo: 0 = más determinista, 1 = más creativo
    model : str
        Nombre del modelo de Gemini a usar
    """
    respuesta = cliente_gemini.models.generate_content(
        model=model,
        contents=prompt,
        config={"temperature": temperature}
    )
    return respuesta.text

In [ ]:
SYSTEM_PROMPT = """Tu objetivo es responder a la pregunta [PREGUNTA] utilizando este contexto [CONTEXTO].

Si la respuesta no se encuentra en el contexto, respondé que no sabés. """


In [ ]:
pregunta = "¿Cual es el nombre del Papa?"

In [ ]:
SYSTEM_PROMPT = SYSTEM_PROMPT.replace("[PREGUNTA]", pregunta)
SYSTEM_PROMPT = SYSTEM_PROMPT.replace("[CONTEXTO]", str(serp_res))

### Aplicando el patrón

Ya tenemos el prompt con contexto inyectado. La siguiente celda lo envía a Gemini y muestra la respuesta.

In [ ]:
respuesta = responder_con_gemini(SYSTEM_PROMPT)
print(respuesta)

### ✎ Para pensar

- ¿Por qué es importante la instrucción "si la respuesta no se encuentra en el contexto, decí que no sabés"?
- ¿Qué pasaría si el modelo pudiera responder tanto con su conocimiento interno como con el contexto?

## Paso 3 — Salida estructurada

Hasta ahora el modelo devuelve texto libre. En aplicaciones reales necesitamos que devuelva **JSON válido** para que nuestro código lo procese.

Con Gemini esto se logra con `response_mime_type="application/json"`: le decimos al modelo que el formato de salida es JSON puro — sin bloques de markdown, sin texto adicional.

In [ ]:
def responder_json_con_gemini(prompt, model="gemini-2.5-flash"):
    """
    Llama a Gemini forzando salida en JSON válido.

    response_mime_type="application/json" garantiza que la respuesta
    sea JSON puro: sin bloques de markdown, sin texto adicional.
    """
    respuesta = cliente_gemini.models.generate_content(
        model=model,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "temperature": 0
        }
    )
    return respuesta.text

### 🐛 Laboratorio de Rotura

El código de abajo le pide un JSON al modelo **sin** ninguna instrucción especial de formato. Antes de ejecutarlo, predecí: ¿el resultado va a ser parseable directamente con `json.loads()`?

Ejecutá la celda siguiente.

- ¿Qué esperabas?
- ¿Qué pasó en realidad?
- ¿Qué te dice eso sobre pedir JSON "a secas" vs usar `response_mime_type`?

No mires la explicación todavía.

In [ ]:
# ── Momento 1 — Pedir JSON sin format forzado ──
prompt_json_roto = """Recomendá 3 series para ver en 2025.
Devolvé un JSON con este formato: {"series": ["serie1", "serie2", "serie3"]}"""

respuesta_sin_format = cliente_gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_json_roto,
    config={"temperature": 0}
)

print("Respuesta del modelo (sin response_mime_type):")
print(respuesta_sin_format.text)
print()

try:
    datos = json.loads(respuesta_sin_format.text)
    print("✓ Parseó bien — el modelo devolvió JSON puro esta vez")
except json.JSONDecodeError as e:
    print(f"✗ Error al parsear: {e}")
    print()
    print("El modelo envolvió el JSON en bloques de markdown (```json...```)")
    print("json.loads() no entiende esos bloques — necesita JSON puro.")

In [ ]:
# ── Momento 2 — Con response_mime_type forzando JSON puro ──
# Esta línea hace que Gemini devuelva SIEMPRE JSON puro, sin excepciones.
respuesta_con_format = responder_json_con_gemini(prompt_json_roto)

print("Con response_mime_type='application/json':")
print(respuesta_con_format)
print()

series = json.loads(respuesta_con_format)
print(f"✓ Parseó correctamente: {series}")
print()
print("◈ response_mime_type es la diferencia entre un sistema frágil y uno robusto.")
print("  Sin él, el modelo puede decidir envolver el JSON cuando le parezca.")

In [ ]:
SYSTEM_PROMPT_JSON = """Tu objetivo es responder a la pregunta [PREGUNTA] utilizando este contexto [CONTEXTO].

El formato de salida debe ser un array JSON con todas las entidades mencionadas,
cada una representada como un único string con el título."""

In [ ]:
pregunta_json = "Recomendaciones de series 2025"
serp_res_json = serp_results(pregunta_json)

prompt_final = SYSTEM_PROMPT_JSON.replace("[PREGUNTA]", pregunta_json)
prompt_final = prompt_final.replace("[CONTEXTO]", str(serp_res_json))

respuesta_json = responder_json_con_gemini(prompt_final)
print("Respuesta estructurada:")
print(respuesta_json)
print()
series_lista = json.loads(respuesta_json)
print(f"Parseado como Python: {series_lista}")

### ✎ Para pensar

- Si el modelo a veces genera texto antes o después del JSON, ¿qué implica eso para un sistema de producción?
- ¿En qué casos preferirías texto libre y en cuáles JSON estructurado?

> **✎ Mentor reversible (bonus).** Hasta acá fuiste quien aprende. Ahora dalo vuelta: tomá el prompt template que armamos con `replace()` y escribí, arriba de esa sección de código, un comentario que le explique a alguien que nunca vio Python qué está pasando ahí y por qué es necesario. Si esa persona imaginaria lo entiende, es porque vos ya lo entendiste de verdad.

## ⛰️ Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **Cutoff date** | El modelo no sabe nada posterior a su fecha de entrenamiento |
| **Alucinación** | El modelo genera respuestas plausibles pero potencialmente falsas |
| **Web search** | Inyectar snippets de búsqueda en el prompt como contexto |
| **Prompt con contexto** | El patrón: `[contexto] + [pregunta]` → respuesta fundamentada |
| **response_mime_type** | Garantiza salida JSON pura sin bloques de markdown |

**Próximo bloque →** Documentos para RAG: cómo cargar PDFs y páginas web y dividirlos en fragmentos listos para vectorizar.

### 🧭 Diario de Navegación

Cerramos mirando hacia adentro, no hacia el código. Respondé en una o dos líneas, para vos:

1. ¿Qué concepto de hoy te costó más encajar en la cabeza? ¿Por qué creés que se resistió?
2. Si tuvieras que explicarle este cuaderno a un colega en lo que dura un viaje en ascensor, ¿qué le dirías?

No hay respuesta correcta. Lo que escribás es el mapa de tu propio recorrido.